# NB03 — Feature Engineering
**Flight Delay Prediction — 2018-2019**

Three feature tiers:
1. **Safe** — per-row derivable (temporal, weather flags, distance)
2. **Operational** — prev-leg delay (tail chaining), rolling histories, congestion + turnaround (pre-computed in NB01)
3. **Target-encoded rates** — fitted on train only

Split: Train Jan–Oct 2018 | Val Nov–Dec 2018 | Test all 2019.

In [53]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv(r"../Data/01_cleaned_data.csv", parse_dates=['DATE'])
print(f'Loaded: {df.shape}')
print(df.columns.tolist())

Loaded: (300000, 24)
['DATE', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY', 'DISTANCE', 'TAIL_NUMBER', 'SCHEDULED_ARRIVAL', 'SCHEDULED_ELAPSED_TIME', 'DAY_OF_WEEK', 'ArrDelay', 'YEAR', 'HOUR', 'TARGET', 'temp', 'precip', 'snowfall', 'wind_speed', 'wind_gusts', 'weather_code', 'ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN']


---
## 1 — Chronological Split

In [54]:
if 'YEAR' not in df.columns:
    df['YEAR'] = df['DATE'].dt.year
df['MONTH'] = df['DATE'].dt.month

df = df.sort_values(['DATE', 'HOUR']).reset_index(drop=True)

train = df[(df['YEAR'] == 2018) & (df['MONTH'] <= 10)].copy()
val   = df[(df['YEAR'] == 2018) & (df['MONTH'] >= 11)].copy()
test  = df[df['YEAR'] == 2019].copy()

print(f'Train: {len(train):,} | {train["DATE"].min().date()} to {train["DATE"].max().date()}')
print(f'Val:   {len(val):,} | {val["DATE"].min().date()} to {val["DATE"].max().date()}')
print(f'Test:  {len(test):,} | {test["DATE"].min().date()} to {test["DATE"].max().date()}')
print(f'\nDelay rates — Train: {train["TARGET"].mean():.3f}  Val: {val["TARGET"].mean():.3f}  Test: {test["TARGET"].mean():.3f}')

Train: 154,458 | 2018-01-01 to 2018-10-31
Val:   45,541 | 2018-11-01 to 2018-12-31
Test:  100,001 | 2019-01-01 to 2019-12-31

Delay rates — Train: 0.189  Val: 0.183  Test: 0.187


---
## 2 — Tier 1: Safe Features

In [55]:
def add_safe_features(df):
    df = df.copy()
    df['MONTH'] = df['DATE'].dt.month
    # DAY_OF_WEEK already exists from NB01 (BTS 1-7 format)
    df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([6, 7]).astype(int)  # BTS: 6=Sat, 7=Sun
    
    # SEASON from EDA monthly delay rates:
    #   High:    Jun(23.6%), Jul(21.2%), Aug(21.6%), Feb(21.0%)
    #   Medium:  May(19.5%), Dec(19.4%) — barely above 18.7%
    #   Low:     Sep(14.5%), Oct(15.6%)
    #   Neutral: Jan(18.0%), Mar(17.9%), Apr(18.1%), Nov(16.7%)
    season_map = {
        1: 1, 2: 3, 3: 1,
        4: 1, 5: 2,
        6: 3, 7: 3, 8: 3,
        9: 0, 10: 0,
        11: 1, 12: 2
    }
    df['SEASON'] = df['MONTH'].map(season_map)
    
    df['LOG_DISTANCE'] = np.log1p(df['DISTANCE'])
    
    # Weather flags from EDA thresholds
    df['IS_FREEZING'] = (df['temp'] < -5).astype(int)       # EDA: +3.8pp
    df['IS_RAINING']  = (df['precip'] > 0).astype(int)      # EDA: +9.5pp
    df['IS_SNOWING']  = (df['snowfall'] > 0).astype(int)    # EDA: +13.0pp
    # EDA: codes 1,2 are BELOW overall rate — excluded
    df['IS_ADVERSE_WEATHER'] = df['weather_code'].isin(
        [51, 53, 55, 61, 63, 65, 71, 73, 75]
    ).astype(int)
    
    df['ROUTE'] = df['ORIGIN_AIRPORT'] + '-' + df['DESTINATION_AIRPORT']
    return df

train = add_safe_features(train)
val   = add_safe_features(val)
test  = add_safe_features(test)
print(f'Safe features done. Columns: {len(train.columns)}')

Safe features done. Columns: 33


---
## 3 — Tier 2: Operational Features
Congestion (ORIGIN_CONGESTION, DEST_CONGESTION) and TURNAROUND_MIN are already in the data from NB01 (pre-computed on full 13M rows).

Prev-leg delay and rolling histories are computed here per-split to avoid leakage.

### 3a — Verify pre-computed features from NB01

In [56]:
# Confirm congestion and turnaround came through from NB01
for col in ['ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN']:
    if col in train.columns:
        corr = train[col].corr(train['TARGET'])
        print(f'  {col}: mean={train[col].mean():.1f} | corr={corr:.4f} | nulls={train[col].isna().sum()}')
    else:
        print(f'  {col}: NOT FOUND — check NB01')

  ORIGIN_CONGESTION: mean=20.8 | corr=-0.0091 | nulls=0
  DEST_CONGESTION: mean=19.9 | corr=-0.0376 | nulls=0
  TURNAROUND_MIN: mean=51.5 | corr=0.0244 | nulls=0


### 3b — Previous Leg Delay (Tail Number Chaining)

In [57]:
def add_prev_leg(df):
    df = df.sort_values(['TAIL_NUMBER', 'DATE', 'HOUR']).reset_index(drop=True)
    df['PREV_LEG_DEP_DELAY'] = df.groupby('TAIL_NUMBER')['DEPARTURE_DELAY'].shift(1)
    if 'ArrDelay' in df.columns:
        df['PREV_LEG_ARR_DELAY'] = df.groupby('TAIL_NUMBER')['ArrDelay'].shift(1)
    df['PREV_LEG_DELAYED'] = (df['PREV_LEG_DEP_DELAY'] >= 15).astype(float)
    df['PREV_LEG_DEP_DELAY'] = df['PREV_LEG_DEP_DELAY'].fillna(-1)
    df['PREV_LEG_DELAYED'] = df['PREV_LEG_DELAYED'].fillna(-1)
    if 'PREV_LEG_ARR_DELAY' in df.columns:
        df['PREV_LEG_ARR_DELAY'] = df['PREV_LEG_ARR_DELAY'].fillna(-1)
    return df

train = add_prev_leg(train)

tv = pd.concat([train, val], ignore_index=True)
tv = add_prev_leg(tv)
val = tv.iloc[len(train):].reset_index(drop=True)

tvt = pd.concat([tv.iloc[:len(train)], val, test], ignore_index=True)
tvt = add_prev_leg(tvt)
test = tvt.iloc[len(train) + len(val):].reset_index(drop=True)

del tv, tvt

print('Prev leg features added.')
for name, split in [('train', train), ('val', val), ('test', test)]:
    has = (split['PREV_LEG_DEP_DELAY'] != -1).mean()
    valid = split[split['PREV_LEG_DEP_DELAY'] != -1]
    corr = valid['PREV_LEG_DEP_DELAY'].corr(valid['TARGET']) if len(valid) > 0 else 0
    print(f'  {name}: {has:.1%} have prev leg | corr={corr:.4f}')

Prev leg features added.
  train: 90.8% have prev leg | corr=0.0416
  val: 90.7% have prev leg | corr=0.0303
  test: 91.4% have prev leg | corr=0.0314


### 3c — Rolling Delay Histories

In [58]:
def add_rolling(df, group_col, delay_col='DEPARTURE_DELAY', window=7):
    col_name = f'{group_col}_ROLL{window}_DELAY'
    df = df.sort_values(['DATE', 'HOUR']).reset_index(drop=True)
    df[col_name] = df.groupby(group_col)[delay_col].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    return df

train = add_rolling(train, 'AIRLINE')
train = add_rolling(train, 'ORIGIN_AIRPORT')

tv = pd.concat([train, val], ignore_index=True)
tv = add_rolling(tv, 'AIRLINE')
tv = add_rolling(tv, 'ORIGIN_AIRPORT')
val = tv.iloc[len(train):].reset_index(drop=True)

tvt = pd.concat([tv.iloc[:len(train)], val, test], ignore_index=True)
tvt = add_rolling(tvt, 'AIRLINE')
tvt = add_rolling(tvt, 'ORIGIN_AIRPORT')
test = tvt.iloc[len(train) + len(val):].reset_index(drop=True)

del tv, tvt

global_mean = train['DEPARTURE_DELAY'].mean()
for split in [train, val, test]:
    split['AIRLINE_ROLL7_DELAY'] = split['AIRLINE_ROLL7_DELAY'].fillna(global_mean)
    split['ORIGIN_AIRPORT_ROLL7_DELAY'] = split['ORIGIN_AIRPORT_ROLL7_DELAY'].fillna(global_mean)

print('Rolling features added.')
for col in ['AIRLINE_ROLL7_DELAY', 'ORIGIN_AIRPORT_ROLL7_DELAY']:
    print(f'  {col}: corr={train[col].corr(train["TARGET"]):.4f}')

Rolling features added.
  AIRLINE_ROLL7_DELAY: corr=0.1587
  ORIGIN_AIRPORT_ROLL7_DELAY: corr=0.1135


---
## 4 — Tier 3: Target-Encoded Rates (train only)

In [59]:
def compute_rate(train_df, group_cols, target_col='TARGET', min_samples=20):
    if isinstance(group_cols, str): group_cols = [group_cols]
    rates = train_df.groupby(group_cols)[target_col].agg(['sum','count'])
    rates['rate'] = rates['sum'] / rates['count']
    rates = rates[rates['count'] >= min_samples]['rate']
    return rates.to_dict(), train_df[target_col].mean()

global_rate = train['TARGET'].mean()
print(f'Global rate: {global_rate:.4f}')

airline_rates, _ = compute_rate(train, 'AIRLINE')
origin_rates, _  = compute_rate(train, 'ORIGIN_AIRPORT')
dest_rates, _    = compute_rate(train, 'DESTINATION_AIRPORT')
route_rates, _   = compute_rate(train, 'ROUTE', min_samples=20)

route_season_rates, _  = compute_rate(train, ['ROUTE', 'SEASON'], min_samples=5)
airline_month_rates, _ = compute_rate(train, ['AIRLINE', 'MONTH'], min_samples=15)
origin_hour_rates, _   = compute_rate(train, ['ORIGIN_AIRPORT', 'HOUR'], min_samples=10)

print(f'Simple: Airlines={len(airline_rates)} Origins={len(origin_rates)} Dests={len(dest_rates)} Routes={len(route_rates)}')
print(f'Interaction: Route×Season={len(route_season_rates)} Airline×Month={len(airline_month_rates)} Origin×Hour={len(origin_hour_rates)}')

Global rate: 0.1888
Simple: Airlines=27 Origins=242 Dests=258 Routes=2438
Interaction: Route×Season=9624 Airline×Month=195 Origin×Hour=2019


In [60]:
def apply_encodings(df, global_rate, airline_rates, origin_rates, dest_rates,
                    route_rates, route_season_rates, airline_month_rates, origin_hour_rates):
    df = df.copy()
    df['AIRLINE_RATE'] = df['AIRLINE'].map(airline_rates).fillna(global_rate)
    df['ORIGIN_RATE']  = df['ORIGIN_AIRPORT'].map(origin_rates).fillna(global_rate)
    df['DEST_RATE']    = df['DESTINATION_AIRPORT'].map(dest_rates).fillna(global_rate)
    
    df['ROUTE_RATE'] = df['ROUTE'].map(route_rates)
    m = df['ROUTE_RATE'].isna()
    df.loc[m, 'ROUTE_RATE'] = df.loc[m, 'ORIGIN_AIRPORT'].map(origin_rates)
    df['ROUTE_RATE'] = df['ROUTE_RATE'].fillna(global_rate)
    
    df['_rs'] = list(zip(df['ROUTE'], df['SEASON']))
    df['ROUTE_SEASON_RATE'] = df['_rs'].map(route_season_rates)
    m = df['ROUTE_SEASON_RATE'].isna()
    df.loc[m, 'ROUTE_SEASON_RATE'] = df.loc[m, 'ROUTE_RATE']
    df.drop(columns='_rs', inplace=True)
    
    df['_am'] = list(zip(df['AIRLINE'], df['MONTH']))
    df['AIRLINE_MONTH_RATE'] = df['_am'].map(airline_month_rates)
    m = df['AIRLINE_MONTH_RATE'].isna()
    df.loc[m, 'AIRLINE_MONTH_RATE'] = df.loc[m, 'AIRLINE_RATE']
    df.drop(columns='_am', inplace=True)
    
    df['_oh'] = list(zip(df['ORIGIN_AIRPORT'], df['HOUR']))
    df['ORIGIN_HOUR_RATE'] = df['_oh'].map(origin_hour_rates)
    m = df['ORIGIN_HOUR_RATE'].isna()
    df.loc[m, 'ORIGIN_HOUR_RATE'] = df.loc[m, 'ORIGIN_AIRPORT'].map(origin_rates).fillna(global_rate)
    df.drop(columns='_oh', inplace=True)
    return df

train = apply_encodings(train, global_rate, airline_rates, origin_rates, dest_rates,
                        route_rates, route_season_rates, airline_month_rates, origin_hour_rates)
val   = apply_encodings(val, global_rate, airline_rates, origin_rates, dest_rates,
                        route_rates, route_season_rates, airline_month_rates, origin_hour_rates)
test  = apply_encodings(test, global_rate, airline_rates, origin_rates, dest_rates,
                        route_rates, route_season_rates, airline_month_rates, origin_hour_rates)

print('Encodings applied.')
for name, split in [('train', train), ('val', val), ('test', test)]:
    rc = [c for c in split.columns if c.endswith('_RATE')]
    print(f'  {name}: {split[rc].isnull().sum().sum()} nulls across {len(rc)} rate cols')

Encodings applied.
  train: 0 nulls across 7 rate cols
  val: 0 nulls across 7 rate cols
  test: 0 nulls across 7 rate cols


---
## 5 — Diagnostics

In [61]:
print('=== Rate Diagnostics ===')
for col in [c for c in train.columns if c.endswith('_RATE')]:
    corr = train[col].corr(train['TARGET'])
    print(f'  {col:>22}: corr={corr:.4f}')

print(f'\nROUTE_SEASON fallback: {(train["ROUTE_SEASON_RATE"] == train["ROUTE_RATE"]).mean():.1%}')

=== Rate Diagnostics ===
            AIRLINE_RATE: corr=0.1154
             ORIGIN_RATE: corr=0.1069
               DEST_RATE: corr=0.1022
              ROUTE_RATE: corr=0.1893
       ROUTE_SEASON_RATE: corr=0.3027
      AIRLINE_MONTH_RATE: corr=0.1495
        ORIGIN_HOUR_RATE: corr=0.2537

ROUTE_SEASON fallback: 11.3%


In [62]:
print('=== Operational Features ===')
op_cols = ['ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN',
           'PREV_LEG_DEP_DELAY', 'PREV_LEG_DELAYED',
           'AIRLINE_ROLL7_DELAY', 'ORIGIN_AIRPORT_ROLL7_DELAY']
if 'PREV_LEG_ARR_DELAY' in train.columns: op_cols.append('PREV_LEG_ARR_DELAY')
for col in op_cols:
    if col in train.columns:
        corr = train[col].corr(train['TARGET'])
        print(f'  {col:>30}: corr={corr:.4f}')

=== Operational Features ===
               ORIGIN_CONGESTION: corr=-0.0091
                 DEST_CONGESTION: corr=-0.0376
                  TURNAROUND_MIN: corr=0.0244
              PREV_LEG_DEP_DELAY: corr=0.0385
                PREV_LEG_DELAYED: corr=0.0498
             AIRLINE_ROLL7_DELAY: corr=0.1587
      ORIGIN_AIRPORT_ROLL7_DELAY: corr=0.1135
              PREV_LEG_ARR_DELAY: corr=0.0407


---
## 6 — Final Feature List & Save

In [63]:
FEATURE_COLS = [
    # Temporal
    'MONTH', 'DAY_OF_WEEK', 'HOUR', 'IS_WEEKEND', 'SEASON',
    # Distance
    'LOG_DISTANCE',
    # Raw weather
    'temp', 'precip', 'snowfall', 'wind_speed', 'wind_gusts', 'weather_code',
    # Weather flags
    'IS_FREEZING', 'IS_RAINING', 'IS_SNOWING', 'IS_ADVERSE_WEATHER',
    # Pre-computed in NB01 (full 13M density)
    'ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN',
    # Tail-number features (per-split)
    'PREV_LEG_DEP_DELAY', 'PREV_LEG_DELAYED',
    # Rolling histories (per-split)
    'AIRLINE_ROLL7_DELAY', 'ORIGIN_AIRPORT_ROLL7_DELAY',
    # Simple rates
    'AIRLINE_RATE', 'ORIGIN_RATE', 'DEST_RATE', 'ROUTE_RATE',
    # Interaction rates
    'ROUTE_SEASON_RATE', 'AIRLINE_MONTH_RATE', 'ORIGIN_HOUR_RATE',
]

# Add prev-leg arrival delay if available
if 'PREV_LEG_ARR_DELAY' in train.columns:
    FEATURE_COLS.append('PREV_LEG_ARR_DELAY')

TARGET_COL = 'TARGET'

print(f'Feature count: {len(FEATURE_COLS)}')
print('\nFeatures:')
for f in FEATURE_COLS:
    print(f'  {f}')

print(f'\nNull check:')
for name, split in [('train', train), ('val', val), ('test', test)]:
    nulls = split[FEATURE_COLS].isnull().sum()
    total = nulls.sum()
    if total > 0:
        print(f'  {name}: {total} nulls')
        print(nulls[nulls > 0])
    else:
        print(f'  {name}: 0 nulls')

Feature count: 31

Features:
  MONTH
  DAY_OF_WEEK
  HOUR
  IS_WEEKEND
  SEASON
  LOG_DISTANCE
  temp
  precip
  snowfall
  wind_speed
  wind_gusts
  weather_code
  IS_FREEZING
  IS_RAINING
  IS_SNOWING
  IS_ADVERSE_WEATHER
  ORIGIN_CONGESTION
  DEST_CONGESTION
  TURNAROUND_MIN
  PREV_LEG_DEP_DELAY
  PREV_LEG_DELAYED
  AIRLINE_ROLL7_DELAY
  ORIGIN_AIRPORT_ROLL7_DELAY
  AIRLINE_RATE
  ORIGIN_RATE
  DEST_RATE
  ROUTE_RATE
  ROUTE_SEASON_RATE
  AIRLINE_MONTH_RATE
  ORIGIN_HOUR_RATE
  PREV_LEG_ARR_DELAY

Null check:
  train: 0 nulls
  val: 0 nulls
  test: 0 nulls


In [64]:
print('=== Feature-Target Correlations (Train) ===')
corrs = train[FEATURE_COLS].corrwith(train[TARGET_COL]).abs().sort_values(ascending=False)
for feat, corr in corrs.items():
    print(f'  {feat:<30} {corr:.4f}')

=== Feature-Target Correlations (Train) ===
  ROUTE_SEASON_RATE              0.3027
  ORIGIN_HOUR_RATE               0.2537
  ROUTE_RATE                     0.1893
  HOUR                           0.1779
  AIRLINE_ROLL7_DELAY            0.1587
  AIRLINE_MONTH_RATE             0.1495
  AIRLINE_RATE                   0.1154
  ORIGIN_AIRPORT_ROLL7_DELAY     0.1135
  ORIGIN_RATE                    0.1069
  DEST_RATE                      0.1022
  weather_code                   0.0927
  wind_gusts                     0.0916
  IS_RAINING                     0.0888
  IS_ADVERSE_WEATHER             0.0888
  SEASON                         0.0711
  precip                         0.0671
  temp                           0.0568
  wind_speed                     0.0509
  PREV_LEG_DELAYED               0.0498
  PREV_LEG_ARR_DELAY             0.0407
  PREV_LEG_DEP_DELAY             0.0385
  DEST_CONGESTION                0.0376
  IS_SNOWING                     0.0290
  LOG_DISTANCE                   0.0

In [65]:
SPLITS_DIR = r"../Data/splits"
os.makedirs(SPLITS_DIR, exist_ok=True)

train[FEATURE_COLS + [TARGET_COL]].to_csv(f'{SPLITS_DIR}/train.csv', index=False)
val[FEATURE_COLS + [TARGET_COL]].to_csv(f'{SPLITS_DIR}/val.csv', index=False)
test[FEATURE_COLS + [TARGET_COL]].to_csv(f'{SPLITS_DIR}/test.csv', index=False)

print('Splits saved.')
print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'Features: {len(FEATURE_COLS)}')

Splits saved.
Train: 154,458 | Val: 45,541 | Test: 100,001
Features: 31
